In [80]:
import torch.nn.functional as F
import matplotlib.pyplot as plt

## 1. RNN-Based Text Generation

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
import torch.nn.functional as F
import re
import requests
import matplotlib.pyplot as plt
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# Dataset:
url = "https://www.gutenberg.org/files/11/11-0.txt"
raw_text = requests.get(url).text
raw_text = re.sub(r'\s+', ' ', raw_text)  # Replace multiple whitespace with a single space
raw_text = raw_text.strip()  # Remove leading and trailing whitespace
raw_text = raw_text[:10000]  # Use only the first 10,000 characters for this example
print(len(raw_text), raw_text[:500])

specials = ["<pad>", "<sos>", "<eos>", "<unk>"]
end_punctuations =  [".", "!", "?"] # Common sentence-ending punctuation marks

# Tokenization (with sentence boundaries):
def tokenize_with_sentence_boundaries(text):
    base_tokens = re.findall(r"\w+|[^\w\s*]", text)
    tokens = ["<sos>"]
    for token in base_tokens:
        tokens.append(token)
        if token in end_punctuations:
            tokens.append("<eos>")
            tokens.append("<sos>")
    if tokens and tokens[-1] == "<sos>":
            tokens.pop()
    if tokens and tokens[-1] != "<eos>":
            tokens.append("<eos>")
    return tokens


processed_text = tokenize_with_sentence_boundaries(raw_text) 
print(f"Number of tokens: {len(processed_text)}, First 500 characters: {processed_text[:500]}")

def encoder(tokens):
    vocab = sorted(set(specials + tokens))
    vocab = {token: idx for idx , token in enumerate(vocab)}
    encoded = [vocab.get(token, vocab["<unk>"]) for token in tokens]
    return encoded, vocab


encoded_text, vocab = encoder(processed_text)
print(f"Vocabulary size: {len(vocab)}, \nFirst 100 tokens in vocabulary: {list(vocab.keys())[:100]} , \nFirst 100 encoded tokens: {encoded_text[:100]}")

# Dataset and DataLoader:
class TextDataset(Dataset):
    def __init__(self, encoded_text, seq_length):
        self.encoded_text = encoded_text
        self.seq_length = seq_length

    def __len__(self):
        return len(self.encoded_text) - self.seq_length

    def __getitem__(self, idx):
        input_seq = self.encoded_text[idx:idx + self.seq_length]
        target_seq = self.encoded_text[idx + 1:idx + self.seq_length + 1]
        return torch.tensor(input_seq, dtype=torch.long), torch.tensor(target_seq, dtype=torch.long)
    
dataset = TextDataset(encoded_text, seq_length=50)
dataloader = DataLoader(dataset, batch_size=32, shuffle=True)

# Hyperparameters:
embedding_dim = 128
hidden_dim = 256
vocab_size = len(vocab)
model = RNNLanguageModel(vocab_size, embedding_dim, hidden_dim, embedding_layer)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)


# Embedding Layer:
embedding_layer = nn.Embedding(vocab_size, embedding_dim)



Device: cpu
10000 *** START OF THE PROJECT GUTENBERG EBOOK 11 *** [Illustration] Alice’s Adventures in Wonderland by Lewis Carroll THE MILLENNIUM FULCRUM EDITION 3.0 Contents CHAPTER I. Down the Rabbit-Hole CHAPTER II. The Pool of Tears CHAPTER III. A Caucus-Race and a Long Tale CHAPTER IV. The Rabbit Sends in a Little Bill CHAPTER V. Advice from a Caterpillar CHAPTER VI. Pig and Pepper CHAPTER VII. A Mad Tea-Party CHAPTER VIII. The Queen’s Croquet-Ground CHAPTER IX. The Mock Turtle’s Story CHAPTER X. The Lobster
Number of tokens: 2509, First 500 characters: ['<sos>', 'START', 'OF', 'THE', 'PROJECT', 'GUTENBERG', 'EBOOK', '11', '[', 'Illustration', ']', 'Alice', '’', 's', 'Adventures', 'in', 'Wonderland', 'by', 'Lewis', 'Carroll', 'THE', 'MILLENNIUM', 'FULCRUM', 'EDITION', '3', '.', '<eos>', '<sos>', '0', 'Contents', 'CHAPTER', 'I', '.', '<eos>', '<sos>', 'Down', 'the', 'Rabbit', '-', 'Hole', 'CHAPTER', 'II', '.', '<eos>', '<sos>', 'The', 'Pool', 'of', 'Tears', 'CHAPTER', 'III', '.', '

### 1.a. Basic RNN Model:

In [64]:
class RNNLanguageModel(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, embedding_layer):
        super().__init__()
        self.embedding = embedding_layer
        self.rnn = nn.RNN(embedding_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, vocab_size)

    def forward(self, x):
        x = self.embedding(x)
        out, _ = self.rnn(x)
        return self.fc(out)

model = RNNLanguageModel(vocab_size, embedding_dim, hidden_dim, embedding_layer).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()


# Training Loop:
num_epochs = 5
for epoch in range(num_epochs):
    model.train()
    total_loss = 0
    for inputs, targets in dataloader:
        inputs, targets = inputs.to(device), targets.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs.view(-1, vocab_size), targets.view(-1))
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    avg_loss = total_loss / len(dataloader)
    print(f"Epoch {epoch+1}/{num_epochs}, Loss: {avg_loss:.4f}")


Epoch 1/5, Loss: 3.9767
Epoch 2/5, Loss: 1.2686
Epoch 3/5, Loss: 0.3712
Epoch 4/5, Loss: 0.1829
Epoch 5/5, Loss: 0.1242


### 1.b. LSTM Model:

In [65]:
class LSTMLanguageModel(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, embedding_layer):
        super().__init__()
        self.embedding = embedding_layer
        self.lstm = nn.LSTM(embedding_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, vocab_size)

    def forward(self, x):
        x = self.embedding(x)
        out, _ = self.lstm(x)
        return self.fc(out)

lstm_model = LSTMLanguageModel(vocab_size, embedding_dim, hidden_dim, embedding_layer).to(device)
lstm_optimizer = torch.optim.Adam(lstm_model.parameters(), lr=0.001)

# Training Loop for LSTM:
for epoch in range(num_epochs):
    lstm_model.train()
    total_loss = 0
    for inputs, targets in dataloader:
        inputs, targets = inputs.to(device), targets.to(device)
        lstm_optimizer.zero_grad()
        outputs = lstm_model(inputs)
        loss = criterion(outputs.view(-1, vocab_size), targets.view(-1))
        loss.backward()
        lstm_optimizer.step()
        total_loss += loss.item()
    avg_loss = total_loss / len(dataloader)
    print(f"LSTM Epoch {epoch+1}/{num_epochs}, Loss: {avg_loss:.4f}")

LSTM Epoch 1/5, Loss: 4.6989
LSTM Epoch 2/5, Loss: 2.1799
LSTM Epoch 3/5, Loss: 0.7830
LSTM Epoch 4/5, Loss: 0.3465
LSTM Epoch 5/5, Loss: 0.2169


In [69]:
# Text Generation (RNN / LSTM):# Build id_to_token mapping (you need this for decoding)
id_to_token = {idx: tok for tok, idx in vocab.items()}

def encode_seed(seed_text, vocab, max_len=None):
    """
    Tokenize seed text the same way as training, then map to ids with <unk> fallback.
    We DO NOT auto-add <eos>/<sos> boundaries here; we just start from the seed.
    """
    # same tokenization pattern you used (excluding '*' as a token)
    seed_tokens = re.findall(r"\w+|[^\w\s*]", seed_text)

    # optional: start with <sos> if you trained with it
    if "<sos>" in vocab:
        seed_tokens = ["<sos>"] + seed_tokens

    unk_id = vocab["<unk>"]
    seed_ids = [vocab.get(t, unk_id) for t in seed_tokens]

    if max_len is not None:
        seed_ids = seed_ids[-max_len:]  # keep last max_len tokens
    return seed_ids, seed_tokens

@torch.no_grad()
def generate_text(
    model,
    vocab,
    id_to_token,
    seed_text="Alice",
    seq_length=50,
    max_new_tokens=120,
    temperature=1.0,
    top_k=30,
    device="cpu",
    stop_on_eos=True
):
    """
    Generates text by repeatedly predicting the next token.
    - temperature: higher => more random, lower => more greedy
    - top_k: sample only from the top_k most likely tokens (set to None to disable)
    """
    model.eval()

    # Encode seed
    seed_ids, _ = encode_seed(seed_text, vocab, max_len=seq_length)
    generated_ids = list(seed_ids)

    eos_id = vocab.get("<eos>", None)

    for _ in range(max_new_tokens):
        # Take last seq_length tokens as context
        context_ids = generated_ids[-seq_length:]
        x = torch.tensor(context_ids, dtype=torch.long, device=device).unsqueeze(0)  # (1, seq_len)

        logits = model(x)                      # (1, seq_len, vocab_size)
        next_logits = logits[0, -1, :]         # (vocab_size,)

        # Temperature
        next_logits = next_logits / max(temperature, 1e-8)

        # Top-k filtering (optional)
        if top_k is not None and top_k > 0:
            top_vals, top_idx = torch.topk(next_logits, k=min(top_k, next_logits.size(-1)))
            probs = F.softmax(top_vals, dim=-1)
            next_id = top_idx[torch.multinomial(probs, num_samples=1)].item()
        else:
            probs = F.softmax(next_logits, dim=-1)
            next_id = torch.multinomial(probs, num_samples=1).item()

        generated_ids.append(next_id)

        if stop_on_eos and eos_id is not None and next_id == eos_id:
            break

    # Decode ids -> tokens
    tokens = [id_to_token[i] for i in generated_ids]

    # Simple detokenization: add spaces between words, but not before punctuation
    out = []
    no_space_before = {".", ",", "!", "?", ":", ";", ")", "]", "}", "\"", "'"}
    no_space_after = {"(", "[", "{", "\"", "'"}
    prev = None
    for t in tokens:
        if t in {"<sos>", "<eos>", "<pad>"}:
            continue
        if not out:
            out.append(t)
        else:
            if t in no_space_before:
                out.append(t)
            elif prev in no_space_after:
                out.append(t)
            else:
                out.append(" " + t)
        prev = t

    return "".join(out)


# Example usage

print("---- RNN generation ----")
print(generate_text(
    model=model,
    vocab=vocab,
    id_to_token=id_to_token,
    seed_text="Alice was",
    seq_length=50,
    max_new_tokens=150,
    temperature=0.9,
    top_k=40,
    device=device,
    stop_on_eos=False   # set True if you want it to stop at <eos>
))

print("\n---- LSTM generation ----")
print(generate_text(
    model=lstm_model,
    vocab=vocab,
    id_to_token=id_to_token,
    seed_text="Alice was",
    seq_length=50,
    max_new_tokens=150,
    temperature=0.9,
    top_k=40,
    device=device,
    stop_on_eos=False
))



---- RNN generation ----
Alice was marked Advice looked flavour IV still nervous considering START talking golden decided you shelves fear OF Presently pop red knelt other mouse looked have plenty jar heads thump burning pocket dipped) she locked funny Either telescope lock air No _through_ still so larger _was_ Who OF miles) funny really Longitude opportunity pocket XI bottle put up leaves s curious _very_, an Little tiny did pop people before lock dream took rather: curious fear indeed once likely hear wondering begin knife couldn marked saw was go After or <unk> drop bit noticed much burning name away fifteen funny field fitted wouldn once but heap wise _never_ without whiskers way re! Now dark sort lit altogether went _would_ thing X ventured later forgotten Why beautifully earnestly littl words opened bright bleeds hot blown people thought looked locks from custard couldn more hot see feeling marked other <unk>

---- LSTM generation ----
Alice was nothing so _very_ remarkable in t

### Analysis:

RNN:
The basic RNN is able to learn short-term word relationships, but it struggles to keep track of information over longer parts of the text. As a result, the generated sentences often become fragmented and lose overall coherence.

LSTM:
The LSTM model produces longer and more coherent passages. It is better at maintaining the structure of the source text because its gating mechanism helps preserve important information across longer sequences.

* One thing to note is we are using a small subset of the book here (10k chars) and a sliding window dataset, so the model can partially memorize. That’s why losses can get very low and generation may copy phrases.

Now we want to compare models with different hyperparameters:

In [81]:
import random
import numpy as np

def train_model(model, dataloader, vocab_size, optimizer, criterion, device, num_epochs=5, clip_norm=1.0):
    model.to(device)
    epoch_losses = []

    for epoch in range(num_epochs):
        model.train()
        total_loss = 0.0

        for inputs, targets in dataloader:
            inputs, targets = inputs.to(device), targets.to(device)

            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs.reshape(-1, vocab_size), targets.reshape(-1))
            loss.backward()

            if clip_norm is not None:
                torch.nn.utils.clip_grad_norm_(model.parameters(), clip_norm)

            optimizer.step()
            total_loss += loss.item()

        avg_loss = total_loss / len(dataloader)
        epoch_losses.append(avg_loss)
        print(f"Epoch {epoch+1}/{num_epochs}, Loss: {avg_loss:.4f}")

    return epoch_losses


def set_seed(seed=0):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
def run_one_experiment(model_type, seq_length, batch_size, embedding_dim, hidden_dim, lr, num_epochs, seed_text="Alice was", seed= 0):
    set_seed(seed)
    # DataLoader depends on seq_length & batch_size
    dataset = TextDataset(encoded_text, seq_length=seq_length)
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

    vocab_size = len(vocab)
    criterion = nn.CrossEntropyLoss()

    # Fresh model each time (fair comparison)
    if model_type.lower() == "rnn":
        emb = nn.Embedding(vocab_size, embedding_dim)
        model = RNNLanguageModel(vocab_size, embedding_dim, hidden_dim, emb)
    else:
        emb = nn.Embedding(vocab_size, embedding_dim)
        model = LSTMLanguageModel(vocab_size, embedding_dim, hidden_dim, emb)

    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    print(f"\n--- {model_type.upper()} | seq={seq_length}, bs={batch_size}, emb={embedding_dim}, hid={hidden_dim}, lr={lr} ---")
    train_model(model, dataloader, vocab_size, optimizer, criterion, device, num_epochs=num_epochs)

    sample = generate_text(
        model=model,
        vocab=vocab,
        id_to_token=id_to_token,
        seed_text=seed_text,
        seq_length=seq_length,
        max_new_tokens=120,
        temperature=0.9,
        top_k=30,
        device=device,
        stop_on_eos=False
    )
    print("\nSample:\n", sample[:600], "...\n")
    return sample

In [77]:
# Investigate the impact of seq_len:
run_one_experiment("lstm", seq_length=30, batch_size=32, embedding_dim=128, hidden_dim=256, lr=1e-3, num_epochs=5)
run_one_experiment("lstm", seq_length=50, batch_size=32, embedding_dim=128, hidden_dim=256, lr=1e-3, num_epochs=5)
run_one_experiment("lstm", seq_length=80, batch_size=32, embedding_dim=128, hidden_dim=256, lr=1e-3, num_epochs=5)


--- LSTM | seq=30, bs=32, emb=128, hid=256, lr=0.001 ---
Epoch 1/5, Loss: 4.8873
Epoch 2/5, Loss: 2.7668
Epoch 3/5, Loss: 1.2679
Epoch 4/5, Loss: 0.6056
Epoch 5/5, Loss: 0.3706

Sample:
 Alice was not a bit hurt, and she jumped up key to get very tired of sitting by her sister on the bank, and of having nothing to do: once or twice she had peeped into the book her sister was reading, but it might end, as she couldn ’ t answer either, it didn ’ t sound at all the right word) “ — but I shall have to ask them what the name of the country, you know. Please, Ma ’ am, is this New Zealand or Australia? ” (and she tried to curtsey as she spoke — fancy _curtseying_ as you ’ ...


--- LSTM | seq=50, bs=32, emb=128, hid=256, lr=0.001 ---
Epoch 1/5, Loss: 4.7857
Epoch 2/5, Loss: 2.4695
Epoch 3/5, Loss: 0.9747
Epoch 4/5, Loss: 0.4207
Epoch 5/5, Loss: 0.2506

Sample:
 Alice was not a bit hurt, and she jumped up on, on to her feet in a moment: she looked up, but it was all dark overhead; before her 

'Alice was not a bit hurt, and she jumped up on the little door, in a dreamy sort of way, “ Do cats eat bats? Do cats eat bats? ” and sometimes, “ Do cats eat bats? ” and sometimes, “ Do bats eat bats? ” (when suddenly, thump! thump! down she came upon a heap of sticks and dry, and the fall was over. Alice was not a bit hurt, and she jumped up on to her feet in a moment: she looked up, but it was all dark'

In [79]:
# Investigate the impact of hidden_dim:
run_one_experiment("lstm", seq_length=50, batch_size=32, embedding_dim=128, hidden_dim=128, lr=1e-3, num_epochs=5)
run_one_experiment("lstm", seq_length=50, batch_size=32, embedding_dim=128, hidden_dim=256, lr=1e-3, num_epochs=5)


--- LSTM | seq=50, bs=32, emb=128, hid=128, lr=0.001 ---
Epoch 1/5, Loss: 5.1625
Epoch 2/5, Loss: 3.3426
Epoch 3/5, Loss: 1.9478
Epoch 4/5, Loss: 1.0498
Epoch 5/5, Loss: 0.6034

Sample:
 Alice was had to get out again that. The Pool of Tears CHAPTER III. The Pool of Tears CHAPTER III. The Queen ’ s Croquet - CHAPTER IX. The Mock Turtle ’ s Story CHAPTER X. A Caucus - Race and a Long Tale CHAPTER IV. A Caucus was nothing so _very_ remarkable. However, and though the fall was now only up and her lessons: she was going to shrink any further she very sleepy, and to see it pop down a telescope? ” Do she said, as for it didn my ...


--- LSTM | seq=50, bs=32, emb=128, hid=256, lr=0.001 ---
Epoch 1/5, Loss: 4.7857
Epoch 2/5, Loss: 2.4695
Epoch 3/5, Loss: 0.9747
Epoch 4/5, Loss: 0.4207
Epoch 5/5, Loss: 0.2506

Sample:
 Alice was not a bit hurt, and she jumped up on, on to her feet in a moment: she looked up, but it was all dark overhead; before her was another long passage, and the White Rabb

'Alice was not a bit hurt, and she jumped up on, on to her feet in a moment: she looked up, but it was all dark overhead; before her was another long passage, and the White Rabbit was still in sight, it might belong to one of the doors of the doors of the hall; but, alas! either the locks were too large, or the key was too small, but at it all seemed quite natural); but when the Rabbit actually _took a watch out of its waistcoat - pocket_, and looked at it, and then hurried on, Alice started to her feet'

In [78]:
# Investigate the impact of learning_rate:
run_one_experiment("lstm", seq_length=50, batch_size=32, embedding_dim=128, hidden_dim=256, lr=3e-4, num_epochs=5)
run_one_experiment("lstm", seq_length=50, batch_size=32, embedding_dim=128, hidden_dim=256, lr=1e-3, num_epochs=5)


--- LSTM | seq=50, bs=32, emb=128, hid=256, lr=0.0003 ---
Epoch 1/5, Loss: 5.6653
Epoch 2/5, Loss: 4.5705
Epoch 3/5, Loss: 3.7464
Epoch 4/5, Loss: 2.9574
Epoch 5/5, Loss: 2.2404

Sample:
 Alice was not a hurry. ” (Alice no she to, “ in a moment, “ after on, ” (she said, this time she had upon a mouse, and behind as the fall passage not. There ’ to get think and about was coming door, but she jumped that a book to of through the house! ” thought Alice was! I should to be shutting herself it, she could for the hot as it was, and there that a little door she saw key to curtsey of the right did you you ever a moment to do: once she was ...


--- LSTM | seq=50, bs=32, emb=128, hid=256, lr=0.001 ---
Epoch 1/5, Loss: 4.7857
Epoch 2/5, Loss: 2.4695
Epoch 3/5, Loss: 0.9747
Epoch 4/5, Loss: 0.4207
Epoch 5/5, Loss: 0.2506

Sample:
 Alice was not a bit hurt, and she jumped up on, on to her feet in a moment: she looked up, but it was all dark overhead; before her was another long passage, and the 

'Alice was not a bit hurt, and she jumped up on, on to her feet in a moment: she looked up, but it was all dark overhead; before her was another long passage, and the White Rabbit was still in sight, it might belong to one of the doors of the doors of the hall; but, alas! either the locks were too large, or the key was too small, but at it all seemed quite natural); but when the Rabbit actually _took a watch out of its waistcoat - pocket_, and looked at it, and then hurried on, Alice started to her feet'

I tested training hyperparameters by changing seq_length, hidden_dim, and learning rate while keeping the dataset and model structure the same. Increasing seq_length from 30 to 80 improved both the training loss and the quality of generated text, likely because the model had access to longer context. Increasing hidden_dim from 128 to 256 significantly improved fluency and reduced repetition, which shows that model capacity matters for text generation. For learning rate, lr=0.0003 underfit within 5 epochs and produced weak text, while lr=0.001 converged faster and produced more coherent outputs.

In [82]:
def plot_two_loss_curves(losses1, losses2, label1="RNN", label2="LSTM", title="Loss Curves Comparison"):
    plt.figure()
    plt.plot(range(1, len(losses1) + 1), losses1, marker="o", label=label1)
    plt.plot(range(1, len(losses2) + 1), losses2, marker="o", label=label2)
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title(title)
    plt.legend()
    plt.grid(True)
    plt.show()

In [83]:
def plot_loss_curve(losses, title="Training Loss Curve"):
    plt.figure()
    plt.plot(range(1, len(losses) + 1), losses, marker="o")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title(title)
    plt.grid(True)
    plt.show()

In [84]:
rnn_losses, rnn_sample = run_one_experiment(
    model_type="rnn", seq_length=50, batch_size=32,
    embedding_dim=128, hidden_dim=256, lr=1e-3, num_epochs=5,
    seed_text="Alice was", seed=0
)

lstm_losses, lstm_sample = run_one_experiment(
    model_type="lstm", seq_length=50, batch_size=32,
    embedding_dim=128, hidden_dim=256, lr=1e-3, num_epochs=5,
    seed_text="Alice was", seed=0
)

plot_two_loss_curves(rnn_losses, lstm_losses, label1="RNN", label2="LSTM",
                     title="RNN vs LSTM Training Loss (seq_length=50)")


--- RNN | seq=50, bs=32, emb=128, hid=256, lr=0.001 ---
Epoch 1/5, Loss: 3.9831
Epoch 2/5, Loss: 1.3110
Epoch 3/5, Loss: 0.3946
Epoch 4/5, Loss: 0.1918
Epoch 5/5, Loss: 0.1280

Sample:
 Alice was not a bit hurt, and she jumped up on to her feet in a moment: she looked up, but it was all dark overhead; before her was another it passage, and the fall _never_ come to an end? “ I wonder how many miles I ’ ve fallen by this time? ” she said aloud. “ I must be getting somewhere near the centre of the earth. Let me see: that would be four thousand miles down, for a jar! ” thought Alice to herself, “ after such a fall as this, I shall think nothing ...



ValueError: too many values to unpack (expected 2)

In [85]:
loss_30, _ = run_one_experiment("lstm", 30, 32, 128, 256, 1e-3, 5, seed_text="Alice was", seed=0)
loss_50, _ = run_one_experiment("lstm", 50, 32, 128, 256, 1e-3, 5, seed_text="Alice was", seed=0)
loss_80, _ = run_one_experiment("lstm", 80, 32, 128, 256, 1e-3, 5, seed_text="Alice was", seed=0)

plt.figure()
plt.plot(range(1, 6), loss_30, marker="o", label="seq_length=30")
plt.plot(range(1, 6), loss_50, marker="o", label="seq_length=50")
plt.plot(range(1, 6), loss_80, marker="o", label="seq_length=80")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("LSTM Loss Curves for Different seq_length Values")
plt.legend()
plt.grid(True)
plt.show()


--- LSTM | seq=30, bs=32, emb=128, hid=256, lr=0.001 ---
Epoch 1/5, Loss: 4.8873
Epoch 2/5, Loss: 2.7668
Epoch 3/5, Loss: 1.2679
Epoch 4/5, Loss: 0.6056
Epoch 5/5, Loss: 0.3706

Sample:
 Alice was not a bit hurt, and she jumped up key to get very tired of sitting by her sister on the bank, and of having nothing to do: once or twice she had peeped into the book her sister was reading, but it might end, as she couldn ’ t answer either, it didn ’ t sound at all the right word) “ — but I shall have to ask them what the name of the country, you know. Please, Ma ’ am, is this New Zealand or Australia? ” (and she tried to curtsey as she spoke — fancy _curtseying_ as you ’ ...



ValueError: too many values to unpack (expected 2)

the plotting tbd

### 2. Seq2Seq Machine Translation with Attention

In [6]:
!wget -q http://www.manythings.org/anki/fra-eng.zip
!unzip -q fra-eng.zip
!ls -lh

total 43M
-rw-r--r-- 1 root root 1.5K Feb 13 06:17 _about.txt
-rw-r--r-- 1 root root 7.9M Feb 13 00:33 fra-eng.zip
-rw-r--r-- 1 root root  35M Feb 13 06:17 fra.txt
drwxr-xr-x 1 root root 4.0K Jan 16 14:24 sample_data


In [8]:
with open("/content/fra.txt", encoding="utf-8") as f:
    lines = f.read().splitlines()

print(len(lines))
print(lines[0])

pairs = []
for line in lines:
    parts = line.split("\t")
    if len(parts) >= 2:
        eng, fra = parts[0].strip(), parts[1].strip()
        pairs.append((eng, fra))

print("pairs:", len(pairs))
print(pairs[:5])

240521
Go.	Va !	CC-BY 2.0 (France) Attribution: tatoeba.org #2877272 (CM) & #1158250 (Wittydev)
pairs: 240521
[('Go.', 'Va !'), ('Go.', 'Marche.'), ('Go.', 'En route !'), ('Go.', 'Bouge !'), ('Hi.', 'Salut !')]
